In [21]:
from pyvis.network import Network
import pickle
import os
import json
import numpy as np
import math

In [113]:
## Thresholds and variables

speaker_seg_tol = 3  ## seconds
span_tol = 3  ## seconds

rel_prob_thresh = 0.25

In [23]:
result_path = "/nfs/data/fakenarratives/202306_corpus/results_pkl/Tagesschau/TV-20230120-2022-1800.webl.h264"

## Read pickles
shots_dict = pickle.load(open(os.path.join(result_path, "transnet_shotdetection.pkl"), "rb"))
asr_dict = pickle.load(open(os.path.join(result_path, "asr.pkl"), "rb"))
nlp_dict = pickle.load(open(os.path.join(result_path, "asr_textnlp.pkl"), "rb"))
event_dict = pickle.load(open(os.path.join(result_path, "eventclassification_vise.pkl"), "rb"))
place_dict = pickle.load(open(os.path.join(result_path, "place_classification.pkl"), "rb"))

In [24]:
print(shots_dict.keys())
print(event_dict.keys())
print(place_dict.keys())
print(asr_dict.keys())
print(nlp_dict.keys())

dict_keys(['inputs', 'outputs', 'pipeline', 'plugins', 'video_file', 'video_id', 'output_data'])
dict_keys(['leaf_node_vector', 'leaf_node_labels', 'time', 'delta_time', 'config', 'args'])
dict_keys(['inputs', 'outputs', 'pipeline', 'plugins', 'video_file', 'video_id', 'output_data'])
dict_keys(['github_repo', 'commit_id', 'parameters', 'video_file', 'output_data'])
dict_keys(['github_repo', 'commit_id', 'parameters', 'video_file', 'output_data'])


In [25]:
print(shots_dict["output_data"].keys())
print(asr_dict["output_data"].keys())

dict_keys(['id', 'version', 'type', 'name', 'ref_id', 'shots'])
dict_keys(['text', 'segments', 'language', 'speaker_segments'])


In [26]:
event_index = event_dict["leaf_node_labels"]
len(event_dict["leaf_node_vector"])

4544

In [27]:
# Create tuples of (time, event), (time, place16)
event_index = event_dict["leaf_node_labels"]
event_list = []
for pred_vector, t in zip(event_dict["leaf_node_vector"], event_dict["time"]):
    label = event_index[np.argmax(pred_vector)]
    event_list.append((label, t))
    
    
place_index = place_dict["output_data"]["probs_places16"]["index"]
place_list = []
for pred_vector, t in zip(place_dict["output_data"]["probs_places16"]["y"], place_dict["output_data"]["probs_places16"]["time"]):
    label = place_index[np.argmax(pred_vector)]
    place_list.append((label, t))

In [28]:
def add_event_nodes(net, event_index):
    for i, event in enumerate(event_index):
        
        net.add_node(event, label=event, title=event)
        
    return net

In [29]:
def add_place_nodes(net, place_index):
    for i, place in enumerate(place_index):
        
        net.add_node(place, label=place, title=place, color="pink")
        
    return net

In [30]:
def add_shot_nodes(net, shots):
    for i, shot in enumerate(shots):
        start_time = shot["start"]
        end_time = shot["end"]

        label = f"Shot {i}"
        title = f"Start: {start_time}, End: {end_time}"

        net.add_node(f"shot_{i}", label=label, title=title, 
                     start_time=start_time, end_time=end_time, color="#ff9999", shape="box")
    
    # Add edges within shot nodes and segment nodes
    for i in range(len(shots) - 1):
        net.add_edge(f"shot_{i}", f"shot_{i+1}")
    
    return net

In [31]:
def get_speaker_turns(speaker_segments):
    for i, segment in enumerate(speaker_segments):
        start_time = round(segment["start"], 2)
        end_time = round(segment["end"], 2)
        if i < len(speaker_segments)-1:
            if speaker_segments[i+1]["start"] - end_time < speaker_seg_tol:
                end_time = round(speaker_segments[i+1]["start"], 2) - 0.04    ## Similar to shots
                
        segment["start"] = start_time
        segment["end"] = end_time
        segment["n_words"] = len(segment["text"].split())
                
    speaker_turns = []
    current_span = None
    for segment in speaker_segments:
        if current_span is None or segment['speaker'] != current_span['speaker']:
            current_span = segment.copy()
            speaker_turns.append(current_span)
        else:
            current_span['end'] = segment['end']
            current_span['text'] += ' ' + segment['text'].strip()
            current_span['n_words'] += segment['n_words']
        
    return speaker_turns, speaker_segments

In [32]:
def add_speakerturn_nodes(net, speaker_turns):         
    for i, turn in enumerate(speaker_turns):
        start_time = turn["start"]
        end_time = turn["end"]
        
        speaker_label = turn["speaker"]
        num_words = turn["n_words"]
        
#         Create the label and title for the node (To see the attribute values and label nodes)
        label = f"Spk. turn {i}"
        title = f"Speaker ID: {speaker_label}, Num Words: {num_words}, Start: {start_time}, End: {end_time}"

        net.add_node(f"spk_turn_{i}", label=label, title=title, 
                     start_time=start_time, end_time=end_time, color="#69cfd3", shape="box")
    
    for i in range(len(speaker_turns) - 1):
        net.add_edge(f"spk_turn_{i}", f"spk_turn_{i+1}")
    
    return net

In [111]:
def add_span_nodes(net, speaker_turns, shots, tolerance=3.0):
    # Combine the two lists and sort by start time
    intervals = sorted(
        [{'start': shot['start'], 'end': shot['end']} for shot in shots] + 
        [{'start': speaker['start'], 'end': speaker['end']} for speaker in speaker_turns],
        key=lambda interval: interval['end']
    )
    
    spans = []
    current_end = 0.0
    
    for i, interval in enumerate(intervals):
        start_time = interval["start"]
        end_time = interval["end"]
        
        if start_time >= current_end and (start_time-current_end) < tolerance: 
            ## For intervals where start time is more than current_end and under tolerance
            span_start = current_end
            span_end = end_time
            if i < len(intervals)-1 and abs(intervals[i+1]["start"] - end_time) < tolerance:
                span_end = intervals[i+1]["start"]
            
            spans.append({'start': span_start, 'end': span_end})
            current_end = span_end
        elif start_time >= current_end and (start_time-current_end) > tolerance: 
            ## Add filler span first if gap is greater than or equal to 3 seconds
            span_start = current_end
            span_end = start_time
            spans.append({'start': span_start, 'end': span_end})    ## Add the filler span first
            
            span_start = start_time
            span_end = end_time
            spans.append({'start': span_start, 'end': span_end})    ## Add current interval now
            
            current_end = span_end
        else:
            ## For intervals ent time is greater than current (to handle shot intervals shorter than speaker turns)
            if end_time > current_end:
                spans.append({'start': current_end, 'end': end_time})
                current_end = end_time

    for i, span in enumerate(spans):
        start_time = span["start"]
        end_time = span["end"]
        
        label = f"Span {i}"
        title = f"Start: {start_time}, End: {end_time}"

        net.add_node(f"span_{i}", label=label, title=title,
                     start_time=start_time, end_time=end_time, color="#66cc66")
        
    for i in range(len(spans) - 1):
        net.add_edge(f"span_{i}", f"span_{i+1}")
        
    return net, spans

In [125]:
def link_attributes_to_spans(net, spans, event_list, place_list):
    for i, span in enumerate(spans):
        start_time = span["start"]
        end_time = span["end"]
        ## Strictly between the spans
#         span_events = list(set([e for e, t in event_list if t > start_time and t < end_time]))

#         for e in span_events:
#             net.add_edge(e, f"span_{i}")

        span_places = list(set([p for p, t in place_list if t > start_time and t < end_time]))

        for p in span_places:
            net.add_edge(p, f"span_{i}")
        
    return net

In [35]:
def add_edges_to_spans(net, shots, speaker_turns, spans):
    # Iterate over the shots
    for i, shot in enumerate(shots):
        start_time = shot["start"]
        end_time = shot["end"]
        
        # Find the spans that overlap with this shot
        overlapping_spans = [f"span_{j}" for j, span in enumerate(spans) 
                             if not (end_time <= span["start"] or start_time >= span["end"])]

        # Add edges from the shot to the overlapping spans
        for span_id in overlapping_spans:
            net.add_edge(f"shot_{i}", span_id)

    # Iterate over the speaker turns
    for i, turn in enumerate(speaker_turns):
        start_time = turn["start"]
        end_time = turn["end"]
        
        # Find the spans that overlap with this speaker segment
        overlapping_spans = [f"span_{j}" for j, span in enumerate(spans) 
                             if not (end_time <= span["start"] or start_time >= span["end"])]

        # Add edges from the speaker segment to the overlapping spans
        for span_id in overlapping_spans:
            net.add_edge(f"spk_turn_{i}", span_id)
            
    return net

In [128]:
# Create a PyVis network
net = Network(notebook=True, directed=True)

options = {
    "physics": {
        "solver": "barnesHut",
        "hierarchicalRepulsion": {
            "nodeDistance": 150
        }
    },
#     "nodes": {
#         "physics": True
#     }
}

net.set_options(json.dumps(options))

## Add place nodes
# net = add_place_nodes(net, place_index)

## Add event nodes
# net = add_event_nodes(net, event_index)

# Limiting to 20 shots
shots = shots_dict["output_data"]["shots"]

# limite to 20 speaker segments
speaker_segments = asr_dict["output_data"]["speaker_segments"]
speaker_turns, speaker_segments = get_speaker_turns(speaker_segments)

net = add_speakerturn_nodes(net, speaker_turns)

net = add_shot_nodes(net, shots)

# Add spans (Smallest temporal element in our graph)
net, spans = add_span_nodes(net, speaker_turns, shots, span_tol)

print("Number of shots:", len(shots))
print("Number of speaker turns:", len(speaker_turns))
print("Number of spans:", len(spans))

net = add_edges_to_spans(net, shots, speaker_turns, spans)

# net = link_attributes_to_spans(net, spans, event_list, place_list)

net.show("graphs/segment_net.html")

Number of shots: 135
Number of speaker turns: 32
Number of spans: 168
graphs/segment_net.html
